<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Accessing DAYMET data**

<font size="3"> Here, we are interested in dayly data for precipitation,.
    
 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)
2. <font size="3"> **Optional**: If running notebook locally, install the conda environment in `environment.yml` file and install conda environment to run notebook.


<span style='color:#ff6666'><font size="5"> **Objectives**

- <font size="3"> Find all relevant <span style='color:#ff6666'>**OPeNDAP**</span> URLs.
- <font size="3"> Using OPeNDAP produced metadata, subset by variable name.
- <font size="3"> Download coordinate data and identify spatial subset
- <font size="3"> Stream data with Pydap+OPeNDAP, downloading only the data of interest.

    

In [1]:
import xarray as xr
import datetime as dt
import earthaccess
import numpy as np

# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via earthaccess and OPeNDAP**

<font size="3.5"> You can authenticate via earthaccess as demonstrated below. You must have a valid EDL account. There are two strategies for authenticating with `earthaccess`:

1. `strategy="interactive"`. This will promt your edl username-password.
2. `strategy="netrc"`. Use this if the notebook is running on an environment where a `.netrc` with your credentials is recoverable. T

<font size="3.5"> Below the default will be `netrc`, assuming the user has executed the notebook **Authenticate.ipynb**. If not, you can change the strategy to `"interactive"`.



In [2]:
auth = earthaccess.login(strategy="netrc", persist=True) # 

# pass Token Authorization to a new Session.
my_session = auth.get_session()

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

<font size="3.5"> Must provide:

- <font size="3.5"> Time range of interest
- <font size="3.5"> Concept collection ID of interest.



In [3]:
daymet_ccid = "C2531982907-ORNL_CLOUD" # 
time_range = [dt.datetime(2014, 5, 15), dt.datetime(2024, 5, 15)] # One month of data

cmr_urls = get_cmr_urls(ccid=daymet_ccid, time_range=time_range, limit=1000) # you can incread the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

################################################ 
 We found a total of  165 OPeNDAP URLS!!!
################################################


/Users/miguelangeljimenezurias/mambaforge/envs/Earthdata2026/lib/python3.12/site-packages/pydap/client.py:1227: UserWarning: Failed to find opendap urls with C2531982907-ORNL_CLOUD. Try again, and make sure the parameters are correct. If you think this is an issue with pydap or the cmr, consider opening an issue on the pydap github repository
  warnings.warn(



### Further filtering

<font size="3.5"> The CMR returns all `precipitation` URLs from DAYMET. However, these are furthern split into three regions:

1. <font size="3.5"> Hawaii
2. <font size="3.5"> Puerto Rico
3. <font size="3.5"> North America (Continental US).

<font size="3.5"> We need to further filter these opendap urls to only retain variables of interest. In this tutorial we are only interested in: `Continental US` we then need to select only the URLs ending with

* `Daymet_Annual_V4R1.daymet_v4_prcp_annttl_na_`



In [ ]:
# filter url to only retain Annual Precip for North America
prcp_na_urls = [url for url in cmr_urls if url.split(".nc")[0].split("Daymet_Annual_V4R1.daymet_v4_")[-1].startswith("prcp_annttl_na")]
prcp_na_urls[:5]

## Subset by variable and by lat/lon values

<font size="3.5"> We will use pydap to download only the metadata of 1 url. Since this is Level 4 data (model output), we only need to figure how to subset 1 file, and we can apply that to all files.

### Area of Interest

<font size="3.5"> MidAtlantic, area surrounds by 
```python
bounding-box = [36.5,-80.3, 40.5,-74 ] # Follows the format: [South, West, North, East]
```


In [ ]:
pyds = open_url(prcp_na_urls[0], protocol="dap4", session=my_session, batch=True)
pyds.tree()

In [ ]:
lon_min, lon_max = -80.3, -74
lat_min, lat_max = 36.5, 40.5

## Lets download data
<font size="3.5"> But only the coordinates latitude and longitude

In [ ]:
%time
lon = pyds['lon'][:].data
lat = pyds["lat"][:].data

In [ ]:
%%time
lon, lat = np.asarray(lon), np.asarray(lat)

In [ ]:
print("######################################## \n longitude array size:", lon.shape, "\n ########################################")

In [ ]:
print("######################################## \n latitude array size:", lat.shape, "\n ########################################")

## Find the indexes that define the area of interest

In [ ]:
# 1) points that fall inside your lat/lon box
mask = (
    (lon >= lon_min) & (lon <= lon_max) &
    (lat >= lat_min) & (lat <= lat_max)
)

rows, cols = np.where(mask)
# indexes below
y0, y1 = rows.min(), rows.max()
x0, x1 = cols.min(), cols.max()



## Double check these values are reasonable

In [ ]:
print(f"#################################################################################### \n Data download will span longitude values: [{lon[y0:y1,x0:x1].ravel().min()}" + ", " + f"{lon[y0:y1,x0:x1].ravel().max()}] \n####################################################################################" )

In [ ]:
print(f"#################################################################################### \n Data download will span latitude values: [{lat[y0:y1,x0:x1].ravel().min()}" + ", " + f"{lat[y0:y1,x0:x1].ravel().max()}] \n ####################################################################################" )

### Define pydap-specific parameters

<font size="3.5"> These are needed to:
- <font size="3.5"> Subset close to remote data (so only data of interest is downloaded)
- <font size="3.5"> Define where to store data in local environment

<font size="3.5"> By default, if no parameter is defined, it will download the entire data and place it in the current directory

In [ ]:
dim_slices = {'/y':(y0,y1), '/x': (x0,x1)} # defines index to subset format: (first, last)
keep_vars = ["/time", "/y", "/x", "/lon", "/lat", "/prcp"] # variables to download
output_path = "data"

# Stream and deserialize data

<font size="3.5"> Pydap will store each remote file into is own individual file (each file will have the same name as that of the source file), instead of aggregating all the data. This is considerable safer (since not all data can be aggregated into single datacube), and enables parallelism.


In [ ]:
%%time
dap_to_netcdf(prcp_na_urls, session=my_session, output_path = output_path, dim_slices=dim_slices, keep_variables=keep_vars)

## Inspect data locally

<font size="3.5"> Once in your local system, you can aggregate the files if needed. In this case all data can be aggregated and is considerable much easier to aggregate these locally, than remotely.



In [ ]:
%%time
ds = xr.open_mfdataset("data/daymet_v4_prcp_annttl_na*", parallel=True, concat_dim='time', combine='nested')

In [ ]:
ds['prcp'].isel(time=1).plot()